In [1]:
import wave
from pathlib import Path

import pandas as pd
import tgt


def load_table(path, participant_col=None):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        if participant_col is not None:
            return pd.read_csv(path, dtype={participant_col: str})
        return pd.read_csv(path)

    if suffix in [".xlsx", ".xls"]:
        if participant_col is not None:
            return pd.read_excel(path, dtype={participant_col: str})
        return pd.read_excel(path)

    raise ValueError(f"Unsupported metadata format: {suffix}")


def get_wav_duration(file_path):
    try:
        with wave.open(str(file_path), "rb") as f:
            return f.getnframes() / float(f.getframerate())
    except Exception:
        return 0.0


def get_valid_wavs_with_textgrids(folder):
    """
    Return only .wav files that have a matching .TextGrid
    in the same folder, matched by stem.
    """
    valid_wavs = []
    for wav_path in folder.glob("*.wav"):
        textgrid_path = wav_path.with_suffix(".TextGrid")
        if textgrid_path.exists():
            valid_wavs.append(wav_path)
    return valid_wavs


def count_words_in_textgrid(textgrid_path):
    """
    Count non-empty word intervals from the MFA 'words' tier.
    """
    try:
        tg = tgt.read_textgrid(str(textgrid_path))
        tier = tg.get_tier_by_name("words")
    except Exception:
        return 0

    count = 0
    for interval in tier.intervals:
        text = str(interval.text).strip()
        if text and text.lower() not in {"sil", "sp", "spn", "<sil>", "silence"}:
            count += 1
    return count


def estimate_cm_windows(num_words, max_context_words=10):
    """
    Estimate number of CM windows produced from one utterance
    with num_words labeled words.
    """
    n = int(num_words) if pd.notna(num_words) else 0
    k = int(max_context_words)
    if n <= 0:
        return 0
    return sum(max(0, n - c + 1) for c in range(1, k + 1))


def get_folder_stats(folder, max_context_words=10):
    """
    Sum duration, word count, and estimated CM windows across all valid wav/TextGrid pairs.
    """
    valid_wavs = get_valid_wavs_with_textgrids(folder)

    total_duration = 0.0
    total_words = 0
    total_estimated_windows = 0

    for wav_path in valid_wavs:
        tg_path = wav_path.with_suffix(".TextGrid")

        dur = get_wav_duration(wav_path)
        n_words = count_words_in_textgrid(tg_path)
        est_windows = estimate_cm_windows(n_words, max_context_words=max_context_words)

        total_duration += dur
        total_words += n_words
        total_estimated_windows += est_windows

    return total_duration, total_words, total_estimated_windows


def normalize_gender(value):
    if pd.isna(value):
        return pd.NA

    s = str(value).strip().lower()
    return s if s else pd.NA


def format_psychosis_score(score_value, scale_name="PANSS10"):
    if pd.isna(score_value):
        return f"{scale_name} NA"

    score_value = float(score_value)
    if score_value.is_integer():
        return f"{scale_name} {int(score_value)}"
    return f"{scale_name} {score_value:.2f}".rstrip("0").rstrip(".")


def get_panss10_columns(metadata_df):
    """
    Find the 10 PANSS10 item columns.

    Preferred:
      - explicit PANSS10 item columns if present
    Fallback:
      - all columns starting with 'PANSS' except aggregate columns
    """
    explicit = [c for c in metadata_df.columns if str(c).startswith("PANSS10")]
    if len(explicit) == 10:
        return explicit

    fallback = [
        c for c in metadata_df.columns
        if str(c).startswith("PANSS")
        and str(c) not in {"PANSS10", "PANSS10G6", "PANSS-Total", "PANSS_Total", "panss-total"}
    ]

    if len(fallback) == 10:
        return fallback

    raise ValueError(
        "Could not identify exactly 10 PANSS10 item columns. "
        f"Found explicit PANSS10*: {explicit}; fallback PANSS*: {fallback}"
    )


def compute_psychosis_columns(meta):
    """
    Adds:
    - label_psychosis from PatientCat:
        1 -> N
        2 -> Y
    - score_psychosis:
        formatted as 'PANSS10 <score>' using PANSS10 total
    - psychosis_remission:
        if label_psychosis == N -> NA
        else Y if all PANSS10 items <= 3, else N
    - gender from Gender column, normalized
    """
    meta = meta.copy()

    if "PatientCat" not in meta.columns:
        raise ValueError("This script requires a 'PatientCat' column.")

    if "Gender" not in meta.columns:
        raise ValueError("This script requires a 'Gender' column.")

    patient_cat = pd.to_numeric(meta["PatientCat"], errors="coerce")

    invalid_mask = ~patient_cat.isin([1, 2]) & patient_cat.notna()
    if invalid_mask.any():
        bad_values = meta.loc[invalid_mask, "PatientCat"].drop_duplicates().tolist()
        raise ValueError(
            f"PatientCat contains invalid values. Expected only 1 or 2, found: {bad_values}"
        )

    meta["label_psychosis"] = patient_cat.map({1: "N", 2: "Y"})
    meta["gender"] = meta["Gender"].apply(normalize_gender)

    panss10_cols = get_panss10_columns(meta)
    panss10 = meta[panss10_cols].apply(pd.to_numeric, errors="coerce")
    panss10_total = panss10.sum(axis=1)

    remission_from_panss = panss10.le(3).all(axis=1).map({True: "Y", False: "N"})
    meta["psychosis_remission"] = remission_from_panss
    meta.loc[meta["label_psychosis"] == "N", "psychosis_remission"] = "NA"

    meta["panss10_total_numeric"] = panss10_total
    meta["score_psychosis"] = meta["panss10_total_numeric"].apply(
        lambda x: format_psychosis_score(x, "PANSS10")
    )

    return meta


def assign_psychosis_and_score(meta, participant_col="Participant_ID"):
    """
    Build a lookup directly from participant_id ->
    (label_psychosis, score_psychosis, psychosis_remission, gender)
    """
    meta = meta.copy()
    meta = compute_psychosis_columns(meta)

    meta[participant_col] = (
        meta[participant_col]
        .astype(str)
        .str.strip()
        .str.strip("'")
        .str.strip('"')
        .str.zfill(3)
    )

    result = meta[
        [participant_col, "label_psychosis", "score_psychosis", "psychosis_remission", "gender"]
    ].copy()
    result = result.dropna(subset=[participant_col])
    result = result.drop_duplicates(subset=[participant_col], keep="first")

    return dict(
        zip(
            result[participant_col],
            zip(
                result["label_psychosis"],
                result["score_psychosis"],
                result["psychosis_remission"],
                result["gender"],
            )
        )
    )


def assign_grouped_balanced_splits(
    df,
    train_ratio=0.80,
    val_ratio=0.10,
    test_ratio=0.10,
    word_weight=12.0,
    utterance_weight=12.0,
    gender_weight=8.0,
    psychosis_weight=4.0,
    psychosis_remission_weight=2.0,
    min_psy_groups_val=6,
    min_psy_groups_test=6,
    min_remission_y_groups_val=3,
    min_remission_y_groups_test=3,
    min_nonpsy_groups_val=2,
    min_nonpsy_groups_test=2,
    soft_cap_factor=1.10,
    overflow_penalty=1e6,
):
    """
    Assign splits by participant_id using utterance count, word count,
    gender, psychosis label, and psychosis remission.

    Depression fields are not used for this dataset and remain NA.
    """
    splits = ["train", "val", "test"]
    split_ratios = {"train": train_ratio, "val": val_ratio, "test": test_ratio}

    gender_labels = sorted(df["gender"].dropna().unique())
    psychosis_labels = sorted(df["label_psychosis"].dropna().unique())
    remission_labels = sorted(
        [x for x in df["psychosis_remission"].dropna().unique() if x in {"Y", "N"}]
    )

    speaker_groups = []
    for participant_id, subdf in df.groupby("participant_id"):
        gender_word = {label: 0.0 for label in gender_labels}
        psy_word = {label: 0.0 for label in psychosis_labels}
        rem_word = {label: 0.0 for label in remission_labels}

        for label, dsub in subdf.groupby("gender"):
            gender_word[label] = float(dsub["num_words"].sum())

        for label, dsub in subdf.groupby("label_psychosis"):
            psy_word[label] = float(dsub["num_words"].sum())

        for label, dsub in subdf.groupby("psychosis_remission"):
            if label in {"Y", "N"}:
                rem_word[label] = float(dsub["num_words"].sum())

        speaker_groups.append(
            {
                "participant_id": participant_id,
                "total_words": float(subdf["num_words"].sum()),
                "total_utterances": float(subdf["num_utterances"].sum()),
                "gender_word": gender_word,
                "psy_word": psy_word,
                "rem_word": rem_word,
                "has_psy": bool((subdf["label_psychosis"] == "Y").any()),
                "has_nonpsy": bool((subdf["label_psychosis"] == "N").any()),
                "has_remission_y": bool((subdf["psychosis_remission"] == "Y").any()),
            }
        )

    speaker_groups.sort(
        key=lambda x: (x["total_words"], x["total_utterances"]),
        reverse=True,
    )

    total_words = float(df["num_words"].sum())
    total_utterances = float(df["num_utterances"].sum())

    total_gender = {
        label: float(df.loc[df["gender"] == label, "num_words"].sum())
        for label in gender_labels
    }
    total_psy = {
        label: float(df.loc[df["label_psychosis"] == label, "num_words"].sum())
        for label in psychosis_labels
    }
    total_rem = {
        label: float(df.loc[df["psychosis_remission"] == label, "num_words"].sum())
        for label in remission_labels
    }

    target_total_words = {
        split: split_ratios[split] * total_words
        for split in splits
    }
    target_total_utterances = {
        split: split_ratios[split] * total_utterances
        for split in splits
    }

    target_gender = {
        split: {
            label: split_ratios[split] * total_gender[label]
            for label in gender_labels
        }
        for split in splits
    }
    target_psy = {
        split: {
            label: split_ratios[split] * total_psy[label]
            for label in psychosis_labels
        }
        for split in splits
    }
    target_rem = {
        split: {
            label: split_ratios[split] * total_rem[label]
            for label in remission_labels
        }
        for split in splits
    }

    current_total_words = {split: 0.0 for split in splits}
    current_total_utterances = {split: 0.0 for split in splits}
    current_gender = {split: {label: 0.0 for label in gender_labels} for split in splits}
    current_psy = {split: {label: 0.0 for label in psychosis_labels} for split in splits}
    current_rem = {split: {label: 0.0 for label in remission_labels} for split in splits}

    psy_group_counts = {"val": 0, "test": 0}
    remission_y_group_counts = {"val": 0, "test": 0}
    nonpsy_group_counts = {"val": 0, "test": 0}

    assignment = {}

    def add_to_split(split, speaker):
        assignment[speaker["participant_id"]] = split

        current_total_words[split] += speaker["total_words"]
        current_total_utterances[split] += speaker["total_utterances"]

        for label in gender_labels:
            current_gender[split][label] += speaker["gender_word"].get(label, 0.0)

        for label in psychosis_labels:
            current_psy[split][label] += speaker["psy_word"].get(label, 0.0)

        for label in remission_labels:
            current_rem[split][label] += speaker["rem_word"].get(label, 0.0)

        if split in {"val", "test"}:
            if speaker["has_psy"]:
                psy_group_counts[split] += 1
            if speaker["has_nonpsy"]:
                nonpsy_group_counts[split] += 1
            if speaker["has_remission_y"]:
                remission_y_group_counts[split] += 1

    def over_soft_cap(split, speaker):
        if split == "train":
            return False

        after_words = current_total_words[split] + speaker["total_words"]
        after_utterances = current_total_utterances[split] + speaker["total_utterances"]

        return (
            after_words > soft_cap_factor * target_total_words[split]
            or after_utterances > soft_cap_factor * target_total_utterances[split]
        )

    def candidate_cost(split, speaker):
        eps = 1e-8
        cost = 0.0

        after_words = current_total_words[split] + speaker["total_words"]
        after_utterances = current_total_utterances[split] + speaker["total_utterances"]

        target_words = target_total_words[split] + eps
        target_utterances = target_total_utterances[split] + eps

        if split != "train":
            if after_words > soft_cap_factor * target_words:
                cost += overflow_penalty * ((after_words / target_words) - soft_cap_factor)

            if after_utterances > soft_cap_factor * target_utterances:
                cost += overflow_penalty * ((after_utterances / target_utterances) - soft_cap_factor)

        cost += word_weight * (((after_words - target_words) / target_words) ** 2)
        cost += utterance_weight * (((after_utterances - target_utterances) / target_utterances) ** 2)

        for label in gender_labels:
            target = target_gender[split][label] + eps
            after = current_gender[split][label] + speaker["gender_word"].get(label, 0.0)
            cost += gender_weight * (((after - target) / target) ** 2)

        for label in psychosis_labels:
            target = target_psy[split][label] + eps
            after = current_psy[split][label] + speaker["psy_word"].get(label, 0.0)
            cost += psychosis_weight * (((after - target) / target) ** 2)

        for label in remission_labels:
            target = target_rem[split][label] + eps
            after = current_rem[split][label] + speaker["rem_word"].get(label, 0.0)
            cost += psychosis_remission_weight * (((after - target) / target) ** 2)

        return cost

    # Phase 1: satisfy psychosis-positive quotas in val/test
    unassigned = []
    for speaker in speaker_groups:
        placed = False

        if speaker["has_psy"]:
            candidates = []
            if psy_group_counts["val"] < min_psy_groups_val and not over_soft_cap("val", speaker):
                candidates.append("val")
            if psy_group_counts["test"] < min_psy_groups_test and not over_soft_cap("test", speaker):
                candidates.append("test")

            if candidates:
                best_split = min(candidates, key=lambda s: candidate_cost(s, speaker))
                add_to_split(best_split, speaker)
                placed = True

        if not placed:
            unassigned.append(speaker)

    # Phase 2: satisfy non-psychosis quotas in val/test
    still_unassigned = []
    for speaker in unassigned:
        placed = False

        if speaker["has_nonpsy"]:
            candidates = []
            if nonpsy_group_counts["val"] < min_nonpsy_groups_val and not over_soft_cap("val", speaker):
                candidates.append("val")
            if nonpsy_group_counts["test"] < min_nonpsy_groups_test and not over_soft_cap("test", speaker):
                candidates.append("test")

            if candidates:
                best_split = min(candidates, key=lambda s: candidate_cost(s, speaker))
                add_to_split(best_split, speaker)
                placed = True

        if not placed:
            still_unassigned.append(speaker)

    # Phase 3: satisfy remission-Y quotas in val/test
    final_unassigned = []
    for speaker in still_unassigned:
        placed = False

        if speaker["has_remission_y"]:
            candidates = []
            if remission_y_group_counts["val"] < min_remission_y_groups_val and not over_soft_cap("val", speaker):
                candidates.append("val")
            if remission_y_group_counts["test"] < min_remission_y_groups_test and not over_soft_cap("test", speaker):
                candidates.append("test")

            if candidates:
                best_split = min(candidates, key=lambda s: candidate_cost(s, speaker))
                add_to_split(best_split, speaker)
                placed = True

        if not placed:
            final_unassigned.append(speaker)

    # Phase 4: assign the rest greedily
    for speaker in final_unassigned:
        candidates = [
            s for s in splits
            if not over_soft_cap(s, speaker) or s == "train"
        ]
        if not candidates:
            candidates = ["train"]

        best_split = min(candidates, key=lambda s: candidate_cost(s, speaker))
        add_to_split(best_split, speaker)

    out = df.copy()
    out["split"] = out["participant_id"].map(assignment)
    return out


def create_balanced_splits_uwo(
    mfa_root,
    metadata_path,
    output_csv,
    participant_col,
    train_ratio=0.80,
    val_ratio=0.10,
    test_ratio=0.10,
):
    dataset_name = "UWO"

    meta = load_table(metadata_path, participant_col=participant_col)
    psychosis_map = assign_psychosis_and_score(
        meta,
        participant_col=participant_col,
    )

    data = []
    root = Path(mfa_root)

    speaker_folders = [folder for folder in root.iterdir() if folder.is_dir()]

    print(f"Scanning speakers... found {len(speaker_folders)} speaker folders")

    for i, folder in enumerate(sorted(speaker_folders), 1):
        pid = folder.name.strip().strip("'").strip('"').zfill(3)

        print(f"[{i}/{len(speaker_folders)}] {pid}")

        psychosis_info = psychosis_map.get(pid)
        if psychosis_info is None:
            print(f"  Skipping {pid}: no metadata match found for participant_id {pid}.")
            continue

        label_psychosis, score_psychosis, psychosis_remission, gender = psychosis_info

        if pd.isna(gender):
            print(f"  Skipping {pid}: missing gender.")
            continue

        duration, num_words, num_utterances = get_folder_stats(folder)

        if duration > 0 and num_words > 0 and num_utterances > 0:
            data.append(
                {
                    "participant_id": pid,
                    "dataset": dataset_name,
                    "base_pid": pid,
                    "gender": gender,
                    "age": "NA",
                    "label_depression": "NA",
                    "score_depression": "NA",
                    "depression_severity": "NA",
                    "label_psychosis": label_psychosis,
                    "psychosis_remission": psychosis_remission,
                    "score_psychosis": score_psychosis,
                    "duration": duration,
                    "num_utterances": int(num_utterances),
                    "num_words": int(num_words),
                }
            )
            print(
                f"  Added {pid}, duration={duration:.2f}s, "
                f"utterances={num_utterances}, words={num_words}"
            )
        else:
            print(f"  Skipping {pid}: no usable .wav/.TextGrid pairs with words found.")

    df = pd.DataFrame(data)

    if df.empty:
        raise ValueError("No usable speaker folders found.")

    print("Balancing speakers...")
    print("Using psychosis rule: PatientCat 1 = N, PatientCat 2 = Y")
    print("Using psychosis remission rule: all PANSS10 items <= 3 -> Y, else N")
    print("Using global balancing across words, utterances, gender, psychosis, and psychosis remission")
    print("Depression fields are not available for UWO and will remain NA")

    out_df = assign_grouped_balanced_splits(
        df,
        train_ratio=train_ratio,
        val_ratio=val_ratio,
        test_ratio=test_ratio,
        word_weight=12.0,
        utterance_weight=12.0,
        gender_weight=8.0,
        psychosis_weight=4.0,
        psychosis_remission_weight=2.0,
        min_psy_groups_val=6,
        min_psy_groups_test=6,
        min_nonpsy_groups_val=2,
        min_nonpsy_groups_test=2,
        min_remission_y_groups_val=3,
        min_remission_y_groups_test=3,
        soft_cap_factor=1.10,
        overflow_penalty=1e6,
    )

    out_df = out_df[
        [
            "participant_id",
            "base_pid",
            "dataset",
            "split",
            "gender",
            "age",
            "label_depression",
            "score_depression",
            "depression_severity",
            "label_psychosis",
            "psychosis_remission",
            "score_psychosis",
            "duration",
            "num_utterances",
            "num_words",
        ]
    ].sort_values(["split", "participant_id"])

    out_df.to_csv(output_csv, index=False)

    print("Done.")

    print("\nCounts by split:")
    print(out_df.groupby("split")["participant_id"].count())

    print("\nCounts by split and gender:")
    print(out_df.groupby(["split", "gender"])["participant_id"].count())

    print("\nCounts by split and psychosis label:")
    print(out_df.groupby(["split", "label_psychosis"])["participant_id"].count())

    print("\nCounts by split and psychosis remission:")
    print(out_df.groupby(["split", "psychosis_remission"])["participant_id"].count())

    print("\nCounts by split, psychosis label, and gender:")
    print(out_df.groupby(["split", "label_psychosis", "gender"])["participant_id"].count())

    print("\nCounts by split, psychosis remission, and gender:")
    print(out_df.groupby(["split", "psychosis_remission", "gender"])["participant_id"].count())

    print("\nDuration by split:")
    print(out_df.groupby("split")["duration"].sum())

    print("\nUtterance counts by split:")
    print(out_df.groupby("split")["num_utterances"].sum())

    print("\nWord counts by split:")
    print(out_df.groupby("split")["num_words"].sum())

    print("\nDuration by split and psychosis label:")
    print(out_df.groupby(["split", "label_psychosis"])["duration"].sum())

    print("\nWord counts by split and psychosis label:")
    print(out_df.groupby(["split", "label_psychosis"])["num_words"].sum())

    print("\nWord counts by split and psychosis remission:")
    print(out_df.groupby(["split", "psychosis_remission"])["num_words"].sum())

    print("\nUtterance counts by split and psychosis label:")
    print(out_df.groupby(["split", "label_psychosis"])["num_utterances"].sum())

    print("\nUtterance counts by split and psychosis remission:")
    print(out_df.groupby(["split", "psychosis_remission"])["num_utterances"].sum())

    print("\nWord count proportions by split:")
    split_words = out_df.groupby("split")["num_words"].sum()
    print((split_words / split_words.sum()).round(4))

    print("\nUtterance count proportions by split:")
    split_utts = out_df.groupby("split")["num_utterances"].sum()
    print((split_utts / split_utts.sum()).round(4))

    print("\nPsychosis speaker counts by split:")
    psy_speaker_counts = (
        out_df.groupby(["split", "participant_id"])["label_psychosis"]
        .first()
        .reset_index()
        .groupby(["split", "label_psychosis"])["participant_id"]
        .nunique()
    )
    print(psy_speaker_counts)

    print("\nRemission-Y speaker counts by split:")
    rem_y_counts = (
        out_df.groupby(["split", "participant_id"])["psychosis_remission"]
        .first()
        .reset_index()
        .assign(remission_y=lambda x: x["psychosis_remission"] == "Y")
        .groupby(["split", "remission_y"])["participant_id"]
        .nunique()
    )
    print(rem_y_counts)

    return out_df

In [ ]:
create_balanced_splits_uwo(
    mfa_root="/work/DISCOURSE/Data/Discourse/AUDIO_CHUNKED/Discourse-UWO/FollowUp",
    metadata_path="/work/DISCOURSE/Data/Discourse/METADATA/DISCOURSE_Metadata_DrEW.xlsx",
    output_csv="UWO_splits.csv",
    participant_col="ID")